Setup + .env + rutas

In [1]:
from pathlib import Path

csv_path      = Path(r"C:\Users\jorge gonzalez\Documents\TFG\Proyecto\data\raw\livvo_ttoo_code_clean.csv")
DATA_PROC_DIR = Path(r"C:\Users\jorge gonzalez\Documents\TFG\Proyecto\data\processed")



for d in [DATA_PROC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Parámetros

In [2]:
USE_COLS = [
    "fechaimport", "date", "codigo_hotel", "codigo_ttoo",
    "roomnights", "bednights", "stock", "ocup", "neto"
]
 
CHUNKSIZE = 200_000
 
STOCK_REAL = {
    "Hotel_1":  94,
    "Hotel_2": 138,
    "Hotel_3": 251,
}
 
PARTS_DIR = DATA_PROC_DIR / "livvo_clean_parts_v2"
PARTS_DIR.mkdir(parents=True, exist_ok=True)

Funciones de limpieza de chunk

In [3]:
import pandas as pd
import numpy as np
 
 
def normalize_chunk(df):
    df["date"]        = pd.to_datetime(df["date"],        errors="coerce")
    df["fechaimport"] = pd.to_datetime(df["fechaimport"], errors="coerce")

    for c in ["codigo_hotel", "codigo_ttoo"]:
        df[c] = df[c].astype(str).str.strip()

    for c in ["roomnights", "bednights", "stock", "ocup", "neto"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df
 
 
def drop_aggregate_lines_intra_chunk(df):
    keys = ["date", "codigo_hotel", "codigo_ttoo"]
    if not all(k in df.columns for k in keys):
        return df
 
    grp  = df.groupby(keys)[["roomnights", "bednights"]].sum()
    grp  = grp.rename(columns={"roomnights": "rn_sum", "bednights": "bn_sum"})
 
    tmp   = df.merge(grp, on=keys, how="left")
    sizes = df.groupby(keys).size().rename("n").reset_index()
    tmp   = tmp.merge(sizes, on=keys, how="left")
 
    cond = (tmp["n"] > 1) & (
        np.isclose(tmp["roomnights"], tmp["rn_sum"], equal_nan=False) |
        np.isclose(tmp["bednights"],  tmp["bn_sum"], equal_nan=False)
    )
 
    return df.loc[~cond.values].copy()
 
 
def apply_stock_rules_chunk(df, stock_real):
    df = df.copy()
    expected = df["codigo_hotel"].map(stock_real)
 
    # UNDER → eliminar
    under = df["stock"] < expected
    df    = df.loc[~under].copy()
 
    # Recalcular expected tras el filtro
    expected = df["codigo_hotel"].map(stock_real)
 
    # OVER → capar
    over = df["stock"] > expected
    df.loc[over, "stock"] = expected.loc[over]
 
    return df

Lectura por chunks + limpieza + parquet

In [4]:
import time
 
try:
    iter_chunks = pd.read_csv(csv_path, usecols=USE_COLS, chunksize=CHUNKSIZE)
except Exception:
    iter_chunks = pd.read_csv(csv_path, usecols=USE_COLS, chunksize=CHUNKSIZE)
 
t0 = time.time()
rows_in = rows_out = 0
 
for i, chunk in enumerate(iter_chunks, 1):
    rows_in += len(chunk)
 
    chunk = normalize_chunk(chunk)
    chunk = drop_aggregate_lines_intra_chunk(chunk)
    chunk = apply_stock_rules_chunk(chunk, STOCK_REAL)
 
    out = PARTS_DIR / f"livvo_clean_part_v2_{i:04d}.parquet"
    chunk.to_parquet(out, index=False)
    rows_out += len(chunk)
 
    if i % 5 == 0:
        print(f"[{i:04d}] {rows_in:,d} → {rows_out:,d}")
 
print("FIN:", time.time() - t0, "s")

[0005] 1,000,000 → 289,660
[0010] 2,000,000 → 584,334
[0015] 3,000,000 → 878,598
[0020] 4,000,000 → 1,172,547
[0025] 5,000,000 → 1,465,417
[0030] 6,000,000 → 1,763,952
[0035] 7,000,000 → 2,066,941
[0040] 8,000,000 → 2,365,380
[0045] 9,000,000 → 2,656,459
[0050] 10,000,000 → 2,947,126
[0055] 11,000,000 → 3,235,105
[0060] 12,000,000 → 3,520,736
[0065] 13,000,000 → 3,804,423
[0070] 14,000,000 → 4,086,626
[0075] 15,000,000 → 4,366,632
[0080] 16,000,000 → 4,644,742
[0085] 17,000,000 → 4,921,776
[0090] 18,000,000 → 5,196,698
[0095] 19,000,000 → 5,457,222
[0100] 20,000,000 → 5,703,198
[0105] 21,000,000 → 5,939,679
[0110] 22,000,000 → 6,177,028
[0115] 23,000,000 → 6,408,414
[0120] 24,000,000 → 6,634,461
[0125] 25,000,000 → 6,854,637
[0130] 26,000,000 → 7,071,995
[0135] 27,000,000 → 7,286,874
[0140] 28,000,000 → 7,499,495
[0145] 29,000,000 → 7,709,253
[0150] 30,000,000 → 7,916,982
[0155] 31,000,000 → 8,122,968
[0160] 32,000,000 → 8,391,557
[0165] 33,000,000 → 8,688,755
[0170] 34,000,000 → 8,985

Unir todos los parquets en uno

In [5]:
from glob import glob
import pyarrow.parquet as pq
 
OUT_FILE = DATA_PROC_DIR / "livvo_clean_all_v2.parquet"
 
parts = sorted(glob(str(PARTS_DIR / "livvo_clean_part_v2_*.parquet")))
 
writer = None
for p in parts:
    table = pq.read_table(p)
    if writer is None:
        writer = pq.ParquetWriter(OUT_FILE, table.schema, compression="snappy")
    writer.write_table(table)
 
writer.close()
OUT_FILE

WindowsPath('C:/Users/jorge gonzalez/Documents/TFG/Proyecto/data/processed/livvo_clean_all_v2.parquet')

In [6]:
df = pd.read_parquet(OUT_FILE)
print(df.shape)


(10108504, 9)
